# Bootstrap: Create All DBSpend360 Tables

Orchestrator notebook that invokes every DDL notebook in `jobs/ddls/` to provision the
full DBSpend360 schema in the requested catalog / schema.

Each child notebook is invoked via `dbutils.notebook.run` with the same `catalog` and
`schema` parameters, so this is the single entry point for setting up a new environment.

**Widgets**
- `catalog`           - target Unity Catalog name
- `schema`            - target schema name within `catalog`
- `timeout_seconds`   - per-notebook timeout (default `600`)

In [ ]:
import json
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("create_all_tables")

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("timeout_seconds", "600", "Per-notebook timeout (seconds)")

In [ ]:
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

try:
    timeout_seconds = int(dbutils.widgets.get("timeout_seconds") or "600")
except ValueError:
    timeout_seconds = 600

if not catalog or not schema:
    raise ValueError("Both `catalog` and `schema` widgets must be set.")

logger.info(f"Provisioning DBSpend360 tables in {catalog}.{schema} (timeout={timeout_seconds}s)")

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [ ]:
DDL_NOTEBOOKS = [
    "dbspend360_audit_log",
    "dbspend360_error_log",
    "dbspend360_covered_workspaces",
    "dbspend360_cloud_cost_explorer",
    "dbspend360_pool_cloud_cost_explorer",
    "dbspend360_dbu_cost",
    "dbspend360_other_cost_breakdown",
    "dbspend360_total_job_spends",
    "dbspend360_all_purpose_dbu_cost",
    "dbspend360_total_all_purpose_spends",
    "dbspend360_pool_dbu_cost",
    "dbspend360_total_pool_spends",
    "dbspend360_pipeline_dbu_cost",
    "dbspend360_total_pipeline_spends",
    "dbspend360_sql_warehouse_dbu_cost",
    "dbspend360_total_sql_warehouse_spends",
]

In [ ]:
results = []
child_args = {"catalog": catalog, "schema": schema}

for nb in DDL_NOTEBOOKS:
    logger.info(f"--> Running ./{nb}")
    try:
        out = dbutils.notebook.run(f"./{nb}", timeout_seconds, child_args)
        results.append({"notebook": nb, "status": "SUCCESS", "output": out})
        logger.info(f"    OK -> {out}")
    except Exception as exc:
        results.append({"notebook": nb, "status": "FAILED", "output": str(exc)})
        logger.exception(f"    FAILED {nb}: {exc}")
        raise

In [ ]:
summary_rows = [
    (r["notebook"], r["status"], r["output"]) for r in results
]
display(spark.createDataFrame(summary_rows, ["notebook", "status", "output"]))

In [ ]:
dbutils.notebook.exit(json.dumps({
    "catalog": catalog,
    "schema": schema,
    "results": results,
}))